## Modélisation - Comparaison de plusieurs algorithmes

Ce notebook compare différents modèles de Machine Learning pour prédire la fraude :
- **Régression logistique**
- **K-Nearest Neighbors (KNN)**
- **Arbres de décision et Forêts aléatoires**
- **Boosting** (AdaBoost / Gradient Boosting)

Pour chaque modèle :
1. Chargement des données preprocessées
2. Entraînement avec optimisation des hyperparamètres (GridSearchCV)
3. Évaluation sur validation (F1-score)
4. Prédictions sur test
5. Export CSV au format Kaggle (`customer_id`, `target_is_fraud`)

**Objectif** : Maximiser le F1-score sur Kaggle.

In [1]:
# Imports nécessaires
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("✅ Imports terminés")

✅ Imports terminés


### 1. Chargement des données preprocessées

On charge les données qui ont été preprocessées dans le notebook `Preprocess_maelle_second.ipynb`.

In [2]:
# Chemins vers les données preprocessées
TRAIN_PREPROCESSED_PATH = "../data/train_preprocessed_v2_V4.csv"
TEST_PREPROCESSED_PATH = "../data/test_preprocessed_v2_V4.csv"
# Chemin vers le fichier test ORIGINAL pour récupérer les customer_id non modifiés
TEST_ORIGINAL_PATH = "../data/kaggle_b2_fraud_test_v4.csv"

# Chargement
train = pd.read_csv(TRAIN_PREPROCESSED_PATH)
test = pd.read_csv(TEST_PREPROCESSED_PATH)

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"\nColonnes train : {train.columns.tolist()[:10]}...")

# Séparation X / y
ID_COL = "customer_id"
TARGET_COL = "target_is_fraud"

X_train = train.drop(columns=[ID_COL, TARGET_COL])
y_train = train[TARGET_COL]
X_test = test.drop(columns=[ID_COL])

# IMPORTANT : Charger les customer_id depuis le fichier ORIGINAL (non preprocessé)
# pour s'assurer qu'ils sont dans le bon format (majuscules, etc.)
try:
    test_original = pd.read_csv(TEST_ORIGINAL_PATH)
    customer_ids_test = test_original[ID_COL]
    print(f"\nCustomer IDs chargés depuis le fichier original : {len(customer_ids_test)} IDs")
    print(f"   Exemple : {customer_ids_test.iloc[0]}")
except FileNotFoundError:
    print(f"\n Fichier original non trouvé, utilisation des IDs du fichier preprocessé")
    customer_ids_test = test[ID_COL]

print(f"\nX_train shape : {X_train.shape}")
print(f"y_train shape : {y_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"\nDistribution de la cible (train) :")
print(y_train.value_counts())
print(f"\nTaux de fraude : {y_train.mean():.4f}")

Train shape : (160000, 50)
Test shape  : (40000, 49)

Colonnes train : ['customer_id', 'target_is_fraud', 'feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7']...

Customer IDs chargés depuis le fichier original : 40000 IDs
   Exemple : CUST_E5RX1BC9II

X_train shape : (160000, 48)
y_train shape : (160000,)
X_test shape  : (40000, 48)

Distribution de la cible (train) :
target_is_fraud
0    146223
1     13777
Name: count, dtype: int64

Taux de fraude : 0.0861


### 1.5. Fonction helper pour exporter au format Kaggle

Fonction pour garantir que les exports sont exactement au format attendu par Kaggle.

In [3]:
def export_submission_kaggle(customer_ids, predictions, filename):
    
    # Créer le DataFrame avec les noms de colonnes EXACTS attendus par Kaggle
    submission = pd.DataFrame({
        'customer_id': customer_ids,  
        'target_is_fraud': predictions 
    })
    
    submission['customer_id'] = submission['customer_id'].astype(str)
    
    submission['target_is_fraud'] = submission['target_is_fraud'].astype(int)
    
    # Export
    filepath = f"../Prédiction/{filename}"
    submission.to_csv(filepath, index=False)
    
    print(f"Exporté : {filename}")
    print(f"Shape : {submission.shape}")
    print(f"Colonnes : {submission.columns.tolist()}")
    print(f"Distribution prédictions : {submission['target_is_fraud'].value_counts().to_dict()}")
    print(f"Exemple customer_id : {submission['customer_id'].iloc[0]}")
    
    return submission

print("Fonction export_submission_kaggle définie")

Fonction export_submission_kaggle définie


### 2. Split train/validation

On garde une partie des données d'entraînement pour la validation et l'optimisation des hyperparamètres.

In [4]:
# Split train/validation (80/20)
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train  # Maintient la proportion de fraude
)

print(f"X_train_split : {X_train_split.shape}")
print(f"X_val         : {X_val.shape}")
print(f"\nTaux de fraude train : {y_train_split.mean():.4f}")
print(f"Taux de fraude val   : {y_val.mean():.4f}")

X_train_split : (128000, 48)
X_val         : (32000, 48)

Taux de fraude train : 0.0861
Taux de fraude val   : 0.0861


### 3. Modèle 1 : Régression Logistique

Régression logistique avec optimisation des hyperparamètres via GridSearchCV.

In [5]:
# Définition de la grille d'hyperparamètres
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [1000]
}

# GridSearchCV avec F1-score comme métrique
lr = LogisticRegression(random_state=42, class_weight='balanced')
grid_search_lr = GridSearchCV(
    lr,
    param_grid_lr,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print(" Recherche des meilleurs hyperparamètres pour Régression Logistique...")
grid_search_lr.fit(X_train_split, y_train_split)

print(f"\nMeilleurs paramètres : {grid_search_lr.best_params_}")
print(f"Meilleur F1-score (CV) : {grid_search_lr.best_score_:.4f}")

# Prédictions sur validation
y_val_pred_lr = grid_search_lr.predict(X_val)
f1_val_lr = f1_score(y_val, y_val_pred_lr)

print(f"\nF1-score sur validation : {f1_val_lr:.4f}")
print("\nRapport de classification :")
print(classification_report(y_val, y_val_pred_lr))

# Prédictions sur test
y_test_pred_lr = grid_search_lr.predict(X_test)

# Export CSV avec format exact Kaggle
submission_lr = export_submission_kaggle(
    customer_ids_test,
    y_test_pred_lr,
    "submission_logistic_regression.csv"
)

 Recherche des meilleurs hyperparamètres pour Régression Logistique...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Meilleurs paramètres : {'C': 0.01, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'saga'}
Meilleur F1-score (CV) : 0.3684

F1-score sur validation : 0.3677

Rapport de classification :
              precision    recall  f1-score   support

           0       0.97      0.77      0.86     29245
           1       0.24      0.78      0.37      2755

    accuracy                           0.77     32000
   macro avg       0.61      0.77      0.61     32000
weighted avg       0.91      0.77      0.82     32000

Exporté : submission_logistic_regression.csv
Shape : (40000, 2)
Colonnes : ['customer_id', 'target_is_fraud']
Distribution prédictions : {0: 28841, 1: 11159}
Exemple customer_id : CUST_E5RX1BC9II


In [12]:
import pickle
import json
import os
from datetime import datetime

# Créer le dossier s'il n'existe pas
os.makedirs("../Models", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Sauvegarder le modèle de régression logistique
model_path = f"../Models/logistic_regression_{timestamp}.pkl"
with open(model_path, "wb") as file:
    pickle.dump(grid_search_lr.best_estimator_, file)
print("Modèle régression logistique sauvegardé ->", model_path)

# Sauvegarder les paramètres
params_path = f"../Models/logistic_params_{timestamp}.json"
with open(params_path, "w") as file:
    json.dump(grid_search_lr.best_params_, file, indent=4)
print("Paramètres sauvegardés ->", params_path)

# Résumé texte
summary_path = f"../Models/logistic_summary_{timestamp}.txt"
with open(summary_path, "w") as file:
    file.write("RÉGRESSION LOGISTIQUE\n")
    file.write("="*30 + "\n")
    file.write(f"Best params: {grid_search_lr.best_params_}\n")
    file.write(f"Best CV score: {grid_search_lr.best_score_:.4f}\n")
    file.write(f"Validation F1: {f1_val_lr:.4f}\n")
    file.write(f"\nRapport de classification:\n")
    file.write(f"Précision classe 1: 0.24\n")
    file.write(f"Recall classe 1: 0.78\n")
    file.write(f"F1-score classe 1: 0.37\n")
print("Résumé sauvegardé ->", summary_path)

Modèle régression logistique sauvegardé -> ../Models/logistic_regression_20260303_133505.pkl
Paramètres sauvegardés -> ../Models/logistic_params_20260303_133505.json
Résumé sauvegardé -> ../Models/logistic_summary_20260303_133505.txt


### 4. Modèle 2 : K-Nearest Neighbors (KNN)

KNN avec optimisation du nombre de voisins et de la métrique de distance.

In [6]:
from sklearn.model_selection import RandomizedSearchCV
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

knn = KNeighborsClassifier()
grid_search_knn = RandomizedSearchCV(
    knn,
    param_grid_knn,
    n_iter=20,  
    cv=3, 
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Recherche des meilleurs hyperparamètres pour KNN...")
grid_search_knn.fit(X_train_split, y_train_split)

print(f"\nMeilleurs paramètres : {grid_search_knn.best_params_}")
print(f"Meilleur F1-score (CV) : {grid_search_knn.best_score_:.4f}")

# Prédictions sur validation
y_val_pred_knn = grid_search_knn.predict(X_val)
f1_val_knn = f1_score(y_val, y_val_pred_knn)

print(f"\nF1-score sur validation : {f1_val_knn:.4f}")
print("\nRapport de classification :")
print(classification_report(y_val, y_val_pred_knn))

# Prédictions sur test
y_test_pred_knn = grid_search_knn.predict(X_test)

# Export CSV avec format exact Kaggle
submission_knn = export_submission_kaggle(
    customer_ids_test,
    y_test_pred_knn,
    "submission_knn.csv"
)

Recherche des meilleurs hyperparamètres pour KNN...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Meilleurs paramètres : {'weights': 'uniform', 'n_neighbors': 3, 'metric': 'euclidean'}
Meilleur F1-score (CV) : 0.1982


  File "C:\Users\maell\AppData\Roaming\Python\Python311\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Program Files\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Program Files\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Program Files\Python311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^



F1-score sur validation : 0.2077

Rapport de classification :
              precision    recall  f1-score   support

           0       0.92      0.98      0.95     29245
           1       0.38      0.14      0.21      2755

    accuracy                           0.91     32000
   macro avg       0.65      0.56      0.58     32000
weighted avg       0.88      0.91      0.89     32000

Exporté : submission_knn.csv
Shape : (40000, 2)
Colonnes : ['customer_id', 'target_is_fraud']
Distribution prédictions : {0: 38726, 1: 1274}
Exemple customer_id : CUST_E5RX1BC9II


In [13]:
import pickle
import json
import os
from datetime import datetime

# Créer le dossier s'il n'existe pas
os.makedirs("../Models", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Sauvegarder le modèle KNN
model_path = f"../Models/knn_{timestamp}.pkl"
with open(model_path, "wb") as file:
    pickle.dump(grid_search_knn.best_estimator_, file)
print("Modèle KNN sauvegardé ->", model_path)

# Sauvegarder les paramètres
params_path = f"../Models/knn_params_{timestamp}.json"
with open(params_path, "w") as file:
    json.dump(grid_search_knn.best_params_, file, indent=4)
print("Paramètres KNN sauvegardés ->", params_path)

# Résumé texte
summary_path = f"../Models/knn_summary_{timestamp}.txt"
with open(summary_path, "w") as file:
    file.write("K-NEAREST NEIGHBORS\n")
    file.write("="*30 + "\n")
    file.write(f"Best params: {grid_search_knn.best_params_}\n")
    file.write(f"Best CV score: {grid_search_knn.best_score_:.4f}\n")
    file.write(f"Validation F1: {f1_val_knn:.4f}\n")
    file.write(f"\nRapport de classification:\n")
    file.write(f"Précision classe 1: 0.38\n")
    file.write(f"Recall classe 1: 0.14\n")
    file.write(f"F1-score classe 1: 0.21\n")
print("Résumé KNN sauvegardé ->", summary_path)

Modèle KNN sauvegardé -> ../Models/knn_20260303_133555.pkl
Paramètres KNN sauvegardés -> ../Models/knn_params_20260303_133555.json
Résumé KNN sauvegardé -> ../Models/knn_summary_20260303_133555.txt


### 5. Modèle 3 : Arbre de Décision

Arbre de décision avec optimisation de la profondeur et autres hyperparamètres.

In [7]:
# Définition de la grille d'hyperparamètres
param_grid_dt = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# GridSearchCV avec F1-score comme métrique
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
grid_search_dt = GridSearchCV(
    dt,
    param_grid_dt,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("Recherche des meilleurs hyperparamètres pour Arbre de Décision...")
grid_search_dt.fit(X_train_split, y_train_split)

print(f"\nMeilleurs paramètres : {grid_search_dt.best_params_}")
print(f"Meilleur F1-score (CV) : {grid_search_dt.best_score_:.4f}")

# Prédictions sur validation
y_val_pred_dt = grid_search_dt.predict(X_val)
f1_val_dt = f1_score(y_val, y_val_pred_dt)

print(f"\nF1-score sur validation : {f1_val_dt:.4f}")
print("\nRapport de classification :")
print(classification_report(y_val, y_val_pred_dt))

# Prédictions sur test
y_test_pred_dt = grid_search_dt.predict(X_test)

# Export CSV avec format exact Kaggle
submission_dt = export_submission_kaggle(
    customer_ids_test,
    y_test_pred_dt,
    "submission_decision_tree.csv"
)

Recherche des meilleurs hyperparamètres pour Arbre de Décision...
Fitting 5 folds for each of 90 candidates, totalling 450 fits

Meilleurs paramètres : {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Meilleur F1-score (CV) : 0.3364

F1-score sur validation : 0.3438

Rapport de classification :
              precision    recall  f1-score   support

           0       0.97      0.77      0.86     29245
           1       0.23      0.72      0.34      2755

    accuracy                           0.76     32000
   macro avg       0.60      0.74      0.60     32000
weighted avg       0.90      0.76      0.81     32000

Exporté : submission_decision_tree.csv
Shape : (40000, 2)
Colonnes : ['customer_id', 'target_is_fraud']
Distribution prédictions : {0: 29147, 1: 10853}
Exemple customer_id : CUST_E5RX1BC9II


In [14]:
import pickle
import json
import os
from datetime import datetime

# Créer le dossier s'il n'existe pas
os.makedirs("../Models", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Sauvegarder le modèle Arbre de Décision
model_path = f"../Models/decision_tree_{timestamp}.pkl"
with open(model_path, "wb") as file:
    pickle.dump(grid_search_dt.best_estimator_, file)
print("Modèle Arbre de Décision sauvegardé ->", model_path)

# Sauvegarder les paramètres
params_path = f"../Models/decision_tree_params_{timestamp}.json"
with open(params_path, "w") as file:
    json.dump(grid_search_dt.best_params_, file, indent=4)
print("Paramètres Arbre de Décision sauvegardés ->", params_path)

# Résumé texte
summary_path = f"../Models/decision_tree_summary_{timestamp}.txt"
with open(summary_path, "w") as file:
    file.write("ARBRE DE DÉCISION\n")
    file.write("="*30 + "\n")
    file.write(f"Best params: {grid_search_dt.best_params_}\n")
    file.write(f"Best CV score: {grid_search_dt.best_score_:.4f}\n")
    file.write(f"Validation F1: {f1_val_dt:.4f}\n")
    file.write(f"\nRapport de classification:\n")
    file.write(f"Précision classe 1: 0.23\n")
    file.write(f"Recall classe 1: 0.72\n")
    file.write(f"F1-score classe 1: 0.34\n")
print("Résumé Arbre de Décision sauvegardé ->", summary_path)

Modèle Arbre de Décision sauvegardé -> ../Models/decision_tree_20260303_133639.pkl
Paramètres Arbre de Décision sauvegardés -> ../Models/decision_tree_params_20260303_133639.json
Résumé Arbre de Décision sauvegardé -> ../Models/decision_tree_summary_20260303_133639.txt


### 6. Modèle 4 : Forêt Aléatoire (Random Forest)

Forêt aléatoire avec optimisation du nombre d'arbres, profondeur, etc.

In [8]:
# Définition de la grille d'hyperparamètres
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

# GridSearchCV avec F1-score comme métrique
rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)
grid_search_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("Recherche des meilleurs hyperparamètres pour Random Forest...")
grid_search_rf.fit(X_train_split, y_train_split)

print(f"\nMeilleurs paramètres : {grid_search_rf.best_params_}")
print(f"Meilleur F1-score (CV) : {grid_search_rf.best_score_:.4f}")

# Prédictions sur validation
y_val_pred_rf = grid_search_rf.predict(X_val)
f1_val_rf = f1_score(y_val, y_val_pred_rf)

print(f"\nF1-score sur validation : {f1_val_rf:.4f}")
print("\nRapport de classification :")
print(classification_report(y_val, y_val_pred_rf))

# Prédictions sur test
y_test_pred_rf = grid_search_rf.predict(X_test)

# Export CSV avec format exact Kaggle
submission_rf = export_submission_kaggle(
    customer_ids_test,
    y_test_pred_rf,
    "submission_random_forest.csv"
)

Recherche des meilleurs hyperparamètres pour Random Forest...
Fitting 5 folds for each of 72 candidates, totalling 360 fits

Meilleurs paramètres : {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Meilleur F1-score (CV) : 0.3900

F1-score sur validation : 0.3807

Rapport de classification :
              precision    recall  f1-score   support

           0       0.94      0.94      0.94     29245
           1       0.37      0.39      0.38      2755

    accuracy                           0.89     32000
   macro avg       0.66      0.67      0.66     32000
weighted avg       0.89      0.89      0.89     32000

Exporté : submission_random_forest.csv
Shape : (40000, 2)
Colonnes : ['customer_id', 'target_is_fraud']
Distribution prédictions : {0: 36319, 1: 3681}
Exemple customer_id : CUST_E5RX1BC9II


In [11]:
import pickle
import json
import os
from datetime import datetime

# Créer le dossier s'il n'existe pas
os.makedirs("../Models", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Sauvegarder le modèle
model_path = f"../Models/random_forest_{timestamp}.pkl"
with open(model_path, "wb") as file:
    pickle.dump(grid_search_rf.best_estimator_, file)
print("Modèle sauvegardé ->", model_path)

# Sauvegarder les paramètres
params_path = f"../Models/params_{timestamp}.json"
with open(params_path, "w") as file:
    json.dump(grid_search_rf.best_params_, file, indent=4)
print("Paramètres sauvegardés ->", params_path)

# Résumé texte
summary_path = f"../Models/summary_{timestamp}.txt"
with open(summary_path, "w") as file:
    file.write(f"Best params: {grid_search_rf.best_params_}\n")
    file.write(f"Best CV score: {grid_search_rf.best_score_:.4f}\n")
    file.write(f"Validation F1: {f1_val_rf:.4f}\n")
print("Résumé sauvegardé ->", summary_path)

Modèle sauvegardé -> ../Models/random_forest_20260303_133417.pkl
Paramètres sauvegardés -> ../Models/params_20260303_133417.json
Résumé sauvegardé -> ../Models/summary_20260303_133417.txt


### 7. Modèle 5 : AdaBoost

AdaBoost avec optimisation du nombre d'estimateurs et du learning rate.

In [ ]:
# Définition de la grille d'hyperparamètres
param_grid_adaboost = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5, 1.0],
    'base_estimator': [
        DecisionTreeClassifier(max_depth=1, random_state=42),
        DecisionTreeClassifier(max_depth=2, random_state=42),
        DecisionTreeClassifier(max_depth=3, random_state=42)
    ]
}

# GridSearchCV avec F1-score comme métrique
adaboost = AdaBoostClassifier(random_state=42)
grid_search_adaboost = GridSearchCV(
    adaboost,
    param_grid_adaboost,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("Recherche des meilleurs hyperparamètres pour AdaBoost...")
grid_search_adaboost.fit(X_train_split, y_train_split)

print(f"\nMeilleurs paramètres : {grid_search_adaboost.best_params_}")
print(f"Meilleur F1-score (CV) : {grid_search_adaboost.best_score_:.4f}")

# Prédictions sur validation
y_val_pred_adaboost = grid_search_adaboost.predict(X_val)
f1_val_adaboost = f1_score(y_val, y_val_pred_adaboost)

print(f"\nF1-score sur validation : {f1_val_adaboost:.4f}")
print("\nRapport de classification :")
print(classification_report(y_val, y_val_pred_adaboost))

# Prédictions sur test
y_test_pred_adaboost = grid_search_adaboost.predict(X_test)

# Export CSV avec format exact Kaggle
submission_adaboost = export_submission_kaggle(
    customer_ids_test,
    y_test_pred_adaboost,
    "submission_adaboost.csv"
)

### 8. Modèle 6 : Gradient Boosting

Gradient Boosting avec optimisation des hyperparamètres.

In [ ]:
# Définition de la grille d'hyperparamètres
param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# GridSearchCV avec F1-score comme métrique
gb = GradientBoostingClassifier(random_state=42)
grid_search_gb = GridSearchCV(
    gb,
    param_grid_gb,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("Recherche des meilleurs hyperparamètres pour Gradient Boosting...")
grid_search_gb.fit(X_train_split, y_train_split)

print(f"\nMeilleurs paramètres : {grid_search_gb.best_params_}")
print(f"Meilleur F1-score (CV) : {grid_search_gb.best_score_:.4f}")

# Prédictions sur validation
y_val_pred_gb = grid_search_gb.predict(X_val)
f1_val_gb = f1_score(y_val, y_val_pred_gb)

print(f"\nF1-score sur validation : {f1_val_gb:.4f}")
print("\nRapport de classification :")
print(classification_report(y_val, y_val_pred_gb))

# Prédictions sur test
y_test_pred_gb = grid_search_gb.predict(X_test)

# Export CSV avec format exact Kaggle
submission_gb = export_submission_kaggle(
    customer_ids_test,
    y_test_pred_gb,
    "submission_gradient_boosting.csv"
)

### 9. Comparaison des modèles

Résumé des performances de tous les modèles sur la validation.

In [ ]:
# Résumé des performances
results_summary = pd.DataFrame({
    'Modèle': [
        'Régression Logistique',
        'KNN',
        'Arbre de Décision',
        'Random Forest',
        'AdaBoost',
        'Gradient Boosting'
    ],
    'F1-score (Validation)': [
        f1_val_lr,
        f1_val_knn,
        f1_val_dt,
        f1_val_rf,
        f1_val_adaboost,
        f1_val_gb
    ],
    'F1-score (CV)': [
        grid_search_lr.best_score_,
        grid_search_knn.best_score_,
        grid_search_dt.best_score_,
        grid_search_rf.best_score_,
        grid_search_adaboost.best_score_,
        grid_search_gb.best_score_
    ]
})

results_summary = results_summary.sort_values('F1-score (Validation)', ascending=False)
print("Comparaison des modèles (triés par F1-score validation) :")
print(results_summary.to_string(index=False))

print("\nMeilleur modèle sur validation :")
best_model_name = results_summary.iloc[0]['Modèle']
best_f1 = results_summary.iloc[0]['F1-score (Validation)']
print(f"   {best_model_name} avec F1-score = {best_f1:.4f}")

print("\nTous les fichiers CSV ont été exportés dans ../Prédiction/")
